# Visualize Converted SFT Parquet

Inspect rows produced by `convert_exported_convos_to_qwen3_vl_zoom_in_sft.py`.
The notebook renders the `MultiTurnSFTDataset` schema directly from parquet.


In [ ]:
from pathlib import Path
import io
import json

import pandas as pd
from PIL import Image
from IPython.display import display, Markdown


In [ ]:
from pathlib import Path
import os

PARQUET_PATH = Path(os.environ.get("SFT_PARQUET", "/path/to/sft.parquet")).expanduser()
assert PARQUET_PATH.exists(), f"Set SFT_PARQUET or edit PARQUET_PATH: {PARQUET_PATH}"


In [ ]:
def to_jsonable(obj):
    if hasattr(obj, 'tolist'):
        return to_jsonable(obj.tolist())
    if hasattr(obj, 'item') and not isinstance(obj, (str, bytes, bytearray)):
        try:
            return to_jsonable(obj.item())
        except Exception:
            pass
    if isinstance(obj, dict):
        return {k: to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(v) for v in obj]
    return obj

def json_default(obj):
    if hasattr(obj, 'tolist'):
        return obj.tolist()
    if hasattr(obj, 'item') and not isinstance(obj, (str, bytes, bytearray)):
        try:
            return obj.item()
        except Exception:
            pass
    return repr(obj)

def dump_json(obj):
    return json.dumps(to_jsonable(obj), ensure_ascii=False, indent=2, default=json_default)

def load_image_payload(item):
    if 'bytes' in item:
        return Image.open(io.BytesIO(item['bytes'])).convert('RGB')
    if 'image' in item:
        path = item['image']
        if path.startswith('file://'):
            path = path[7:]
        return Image.open(path).convert('RGB')
    raise ValueError(f'Unknown image payload: {item}')

def render_text_with_images(text, images, image_ptr):
    chunks = text.split('<image>')
    for j, chunk in enumerate(chunks):
        if chunk:
            display(Markdown(f'```text\n{chunk}\n```'))
        if j < len(chunks) - 1:
            if image_ptr < len(images):
                display(images[image_ptr])
                image_ptr += 1
            else:
                display(Markdown('`[missing image]`'))
    return image_ptr

def display_converted_row(idx: int):
    row = df.iloc[idx]
    images = [load_image_payload(x) for x in row['images']]
    image_ptr = 0

    display(Markdown(f'## Row {idx}'))
    display(Markdown(f'`message_loss_mask`: {to_jsonable(row["message_loss_mask"])}'))

    for i, msg in enumerate(row['messages']):
        role = msg.get('role')
        content = msg.get('content')
        display(Markdown(f'### {i}. {role}'))

        tool_calls = msg.get('tool_calls')
        if tool_calls:
            display(Markdown('**tool_calls**'))
            display(Markdown(f'```json\n{dump_json(tool_calls)}\n```'))

        if isinstance(content, list):
            for part in content:
                part_type = part.get('type')
                if part_type == 'text':
                    text = part.get('text', '')
                    if text:
                        display(Markdown(f'```text\n{text}\n```'))
                elif part_type == 'image':
                    if image_ptr < len(images):
                        display(images[image_ptr])
                        image_ptr += 1
                    else:
                        display(Markdown('`[missing image]`'))
                else:
                    display(Markdown(f'```json\n{dump_json(part)}\n```'))
            continue

        if isinstance(content, str):
            image_ptr = render_text_with_images(content, images, image_ptr)
            if i == 0 and role == 'system':
                display(Markdown('**tools schema**'))
                display(Markdown(f'```json\n{dump_json(row["tools"])}\n```'))
            continue

        display(Markdown(f'```json\n{dump_json(content)}\n```'))
        if i == 0 and role == 'system':
            display(Markdown('**tools schema**'))
            display(Markdown(f'```json\n{dump_json(row["tools"])}\n```'))


In [ ]:
import random
idx = random.randint(0, len(df) - 1)
display_converted_row(idx)